# [과제3] 특성 공학(Feature Engineering) 파이프라인 구현 및 성능 비교 실험
This project was developed with partial assistance from ChatGPT.  

ChatGPT was used for feature engineering ideas, pipeline structure,  

code organization, and report writing support.  

Final implementation, execution, and result analysis were performed manually.

## 0. Colab 파일 업로드

In [ ]:
from google.colab import files
uploaded = files.upload()

## 1. 라이브러리 설치 및 불러오기

In [ ]:
!pip install xgboost -q

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

## 2. STEP 01 데이터 준비

In [ ]:
df = pd.read_csv("train.csv")

print("데이터 크기:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all"))

### 타겟 변수 정의

In [ ]:
target = "Survived"

print(df[target].value_counts())
print(df[target].value_counts(normalize=True))

### 컬럼 설명 표

In [ ]:
column_desc = pd.DataFrame({
    "Column": df.columns,
    "Description": [
        "승객 ID",
        "생존 여부, 0=사망, 1=생존",
        "객실 등급, 1=1등석, 2=2등석, 3=3등석",
        "승객 이름",
        "성별",
        "나이",
        "형제/배우자 수",
        "부모/자녀 수",
        "티켓 번호",
        "운임 요금",
        "객실 번호",
        "탑승 항구"
    ]
})

display(column_desc)

## 3. STEP 02 EDA

### 3-1. 결측치 비율 분석

In [ ]:
missing_table = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_ratio": df.isnull().mean() * 100
}).sort_values(by="missing_ratio", ascending=False)

display(missing_table)

### 3-2. 타겟 변수 분포 확인

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Survived")
plt.title("Target Distribution: Survived")
plt.show()

### 3-3. Histogram

In [ ]:
numeric_cols = ["Age", "Fare", "SibSp", "Parch"]

df[numeric_cols].hist(figsize=(12,8), bins=30)
plt.suptitle("Numeric Feature Distributions")
plt.show()

### 3-4. Boxplot: 이상치 탐색

In [ ]:
for col in ["Age", "Fare"]:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, y=col)
    plt.title(f"Boxplot of {col}")
    plt.show()

### 3-5. Countplot / Barplot

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Sex", hue="Survived")
plt.title("Survival by Sex")
plt.show()

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Pclass", hue="Survived")
plt.title("Survival by Pclass")
plt.show()

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Embarked", hue="Survived")
plt.title("Survival by Embarked")
plt.show()

### 3-6. 상관관계 Heatmap

In [ ]:
corr_df = df.select_dtypes(include=["int64", "float64"])

plt.figure(figsize=(8,6))
sns.heatmap(corr_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 4. STEP 03 파생 변수 생성

In [ ]:
def create_features(data):
    data = data.copy()

    # 가족 수
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1

    # 혼자 탑승 여부
    data["IsAlone"] = np.where(data["FamilySize"] == 1, 1, 0)

    # 1인당 요금
    data["FarePerPerson"] = data["Fare"] / data["FamilySize"]

    # 이름에서 호칭 추출
    data["Title"] = data["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

    # 나이 그룹
    data["AgeGroup"] = pd.cut(
        data["Age"],
        bins=[0, 12, 19, 59, 100],
        labels=["Child", "Teen", "Adult", "Senior"]
    )

    return data

In [ ]:
df_fe = create_features(df)

display(df_fe[["FamilySize", "IsAlone", "FarePerPerson", "Title", "AgeGroup"]].head())

## 5. 모델 학습용 데이터 구성

In [ ]:
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]

X = df_fe.drop(columns=[target] + drop_cols)
y = df_fe[target]

print(X.shape)
display(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 6. 실험 함수 정의

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    result = {
        "Experiment": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc
    }

    return result

## 7. Base 실험

In [ ]:
base_df = df_fe.copy()

base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "FarePerPerson"]

base_data = base_df[base_features + [target]].dropna()

X_base = base_data.drop(columns=[target])
y_base = base_data[target]

X_base_train, X_base_test, y_base_train, y_base_test = train_test_split(
    X_base, y_base,
    test_size=0.2,
    random_state=42,
    stratify=y_base
)

base_model = RandomForestClassifier(random_state=42)

base_result = evaluate_model(
    "Base",
    base_model,
    X_base_train,
    X_base_test,
    y_base_train,
    y_base_test
)

base_result

## 8. Exp-1: Mean + One-Hot + StandardScaler + Feature Selection X

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

In [ ]:
preprocess_exp1 = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler())
        ]), numeric_features),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

exp1_model = Pipeline([
    ("preprocess", preprocess_exp1),
    ("model", RandomForestClassifier(random_state=42))
])

exp1_result = evaluate_model(
    "Exp-1",
    exp1_model,
    X_train,
    X_test,
    y_train,
    y_test
)

exp1_result

## 9. Exp-2: Median + Label Encoding + MinMaxScaler + Feature Selection O

In [ ]:
preprocess_exp2 = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler())
        ]), numeric_features),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), categorical_features)
    ]
)

exp2_model = Pipeline([
    ("preprocess", preprocess_exp2),
    ("feature_selection", SelectKBest(score_func=f_classif, k=10)),
    ("model", RandomForestClassifier(random_state=42))
])

exp2_result = evaluate_model(
    "Exp-2",
    exp2_model,
    X_train,
    X_test,
    y_train,
    y_test
)

exp2_result

## 10. Exp-3: Most Frequent + One-Hot + RobustScaler + Feature Selection O

In [ ]:
preprocess_exp3 = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("scaler", RobustScaler())
        ]), numeric_features),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

exp3_model = Pipeline([
    ("preprocess", preprocess_exp3),
    ("feature_selection", SelectKBest(score_func=f_classif, k=10)),
    ("model", RandomForestClassifier(random_state=42))
])

exp3_result = evaluate_model(
    "Exp-3",
    exp3_model,
    X_train,
    X_test,
    y_train,
    y_test
)

exp3_result

## 11. 실험 비교표 작성

In [ ]:
results = pd.DataFrame([
    base_result,
    exp1_result,
    exp2_result,
    exp3_result
])

display(results)

In [ ]:
experiment_conditions = pd.DataFrame({
    "Experiment": ["Base", "Exp-1", "Exp-2", "Exp-3"],
    "Missing Value": ["None", "Mean", "Median", "Most Frequent"],
    "Encoding": ["None", "One-Hot", "Label/Ordinal", "One-Hot"],
    "Scaling": ["None", "StandardScaler", "MinMaxScaler", "RobustScaler"],
    "Feature Selection": ["None", "X", "O", "O"]
})

display(experiment_conditions)

## 12. 모델 2개 이상 비교

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    )
}

model_results = []

for model_name, clf in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess_exp3),
        ("feature_selection", SelectKBest(score_func=f_classif, k=10)),
        ("model", clf)
    ])

    result = evaluate_model(
        model_name,
        pipe,
        X_train,
        X_test,
        y_train,
        y_test
    )

    model_results.append(result)

model_results_df = pd.DataFrame(model_results)
display(model_results_df)

## 13. Feature Importance 확인

In [ ]:
rf_pipe = Pipeline([
    ("preprocess", preprocess_exp3),
    ("model", RandomForestClassifier(random_state=42))
])

rf_pipe.fit(X_train, y_train)

In [ ]:
preprocessor = rf_pipe.named_steps["preprocess"]
rf_model = rf_pipe.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

display(importance_df.head(15))

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=importance_df.head(15), x="Importance", y="Feature")
plt.title("Top 15 Feature Importance")
plt.show()

## 14. Feature Selection 전/후 성능 비교

In [ ]:
without_fs = Pipeline([
    ("preprocess", preprocess_exp3),
    ("model", RandomForestClassifier(random_state=42))
])

with_fs = Pipeline([
    ("preprocess", preprocess_exp3),
    ("feature_selection", SelectKBest(score_func=f_classif, k=10)),
    ("model", RandomForestClassifier(random_state=42))
])

fs_results = []

fs_results.append(evaluate_model(
    "Without Feature Selection",
    without_fs,
    X_train,
    X_test,
    y_train,
    y_test
))

fs_results.append(evaluate_model(
    "With Feature Selection",
    with_fs,
    X_train,
    X_test,
    y_train,
    y_test
))

fs_results_df = pd.DataFrame(fs_results)
display(fs_results_df)

## 15. GridSearchCV 가산점 코드

In [ ]:
grid_pipe = Pipeline([
    ("preprocess", preprocess_exp3),
    ("feature_selection", SelectKBest(score_func=f_classif)),
    ("model", RandomForestClassifier(random_state=42))
])

param_grid = {
    "feature_selection__k": [8, 10, 12, "all"],
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 7, None],
    "model__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    grid_pipe,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)

In [ ]:
grid_result = evaluate_model(
    "GridSearchCV Best Model",
    grid_search.best_estimator_,
    X_train,
    X_test,
    y_train,
    y_test
)

grid_result

## 16. 최종 결과 통합

In [ ]:
final_results = pd.concat([
    results,
    model_results_df,
    fs_results_df,
    pd.DataFrame([grid_result])
], ignore_index=True)

display(final_results)